In [ ]:
!pip install transformers torchvision pillow opencv-python yt-dlp

In [ ]:
import os

url = "https://www.youtube.com/watch?v=BB49x_uMlGA"

os.system(f"yt-dlp -f mp4 -o video.mp4 {url}")

In [ ]:
import cv2

def extract_frames(video_path, interval=30):
    cap = cv2.VideoCapture(video_path)
    frames = []
    count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if count % interval == 0:
            frames.append(frame)

        count += 1

    cap.release()
    return frames

frames = extract_frames("video.mp4", interval=60)
print(f"Extracted {len(frames)} frames")

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)

In [ ]:
def caption_frame(frame):
    image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    inputs = processor(image, return_tensors="pt").to(device)

    out = model.generate(**inputs, max_new_tokens=30)
    caption = processor.decode(out[0], skip_special_tokens=True)
    return caption

captions = []

for i, frame in enumerate(frames):
    cap = caption_frame(frame)
    print(f"Frame {i}: {cap}")
    captions.append(cap)

In [ ]:
from collections import Counter

# simple aggregation
summary = Counter(captions).most_common(5)

print("\n--- Final Video Description ---")
for s, _ in summary:
    print("-", s)